In [54]:
import warnings
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import schemdraw
import schemdraw.elements as elm
from matplotlib.axes import Axes

from drawing_utils import Arrow, Point

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({"text.usetex": True})

# Three-Phase Power

## Comparison of *n*-phase systems

Why three-phase power? Why not two-phase power, or four? This section will compare systems with an arbitrary number of phases, $n$. We will determine the advantages and disadvantages of systems with different numbers of phases.

<!-- Just because the voltages sum to zero, doesn't mean that the currents do. -->

**Single-phase power**

$$
v_a(t) = |V| \cos \omega t
$$

$$
p_{\, \text{tot.}}(t)
= p_a(t)
= \frac{|V|^2}{R}
= \frac{|V|^2}{R} \cos^2 \omega t
= \underbrace{\, \frac{1}{2} \frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}} + \frac{1}{2} \frac{|V|^2}{R} \cos 2 \omega t
$$ (single_phase_p)

$$
i_{\, \text{tot.}}(t)
= i_a(t)
= \frac{|V|}{R} \cos \omega t \neq 0 \implies i_n(t)
$$

**Two-phase power**

$$
\begin{aligned}
& v_a(t) = |V| \cos \omega t \\
& v_b(t) = |V| \cos (\omega t + \pi / 2)
\end{aligned}
$$

$$
p_{\, \text{tot.}}(t)
= p_a(t) + p_b(t)
= \frac{V_a^2 + V_b^2}{R}
= \frac{(|V| \cos \omega t)^2 + \left( |V| \cos \! \left( \omega t + \frac{\pi}{2} \right) \right)^2}{R}
= \frac{|V|^2 ((\cos \omega t)^2 + (- \sin \omega t)^2)}{R}
% Using the Pythagorean identity:
= \underbrace{\frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}}
$$

$$
\begin{aligned}
i_{\, \text{tot.}}(t)
= i_a(t) + i_b(t)
& = |V| \cos \omega t + |V| \cos \! \left( \omega t + \frac{\pi}{2} \right) \\
& = \sqrt{2} \, \frac{|V|}{R} \cos \! \left( \omega t + \frac{\pi}{4} \right)
\end{aligned}
$$

**Three-phase power**

$$
\begin{aligned}
& v_a(t) = |V| \cos \omega t \\
& v_b(t) = |V| \cos (\omega t + \pi / 3) \\
& v_c(t) = |V| \cos (\omega t + 2 \pi / 3)
\end{aligned}
$$

$$
p_{\, \text{tot.}}(t)
= p_a(t) + p_b(t) + p_c(t)
= \underbrace{\frac{3}{2} \frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}}
$$

Four-phase power is just 2x two-phase power

More than four phases

| # of phases    | Min. # of lines required | Power, $p_{\, \text{tot.}}(t)$        | Avg. power, $\bar{p}_{\, \text{tot.}}$ | Avg. power per line  |
|----------------|--------------------------|---------------------------------------|----------------------------------------|----------------------|
| 1              | 2                        | Sinusoidal (Eq. {eq}`single_phase_p`) | $= \|V\|^2 / \, 2 R$                   | $= \|V\|^2 / \, 4 R$ |
| 2              | 3                        | Constant                              | $= \|V\|^2 / R$                        | $= \|V\|^2 / \, 3 R$ |
| 3              | 3                        | Constant                              | $= 3 \|V\|^2 / \, 2 R$                 | $= \|V\|^2 / \, 2 R$ |
| 4              | 4                        | Constant                              | $= 2 \|V\|^2 / R$                      | $= \|V\|^2 / \, 2 R$ |
| Any $n \geq 3$ | $n$                      | Constant                              | $= n \|V\|^2 / \, 2 R$                 | $= \|V\|^2 / \, 2 R$ |

In [ ]:
def draw(ax0: Axes, ax1: Axes, ax2: Axes, ax3: Axes, angles: list[float]) -> None:
    n_phases = len(angles)
    title = f"{n_phases} phases" if n_phases > 1 else "Single phase"
    ax0.set_title(rf"\small \rm {title}", loc="left")

    ax0.set_aspect("equal")
    ax0.set_axis_off()

    ax0.set_xlim((-5, 5))
    ax0.set_ylim((-4, 5))

    with schemdraw.Drawing(canvas=ax0) as d:
        d.config(lw=0.5)
        origin = (0.0, (-1.5 if n_phases == 1 else 0.0))
        elm.Dot().at(origin)
        if n_phases < 3:
            elm.Ground()
        for i, angle in enumerate(angles):
            elm.Resistor().at(origin).length(4).theta(np.rad2deg(angle)).label(
                r"\small $R$"
            )
            phase_letter = "abcde"[i]
            elm.Dot().label(rf"\small $v_{phase_letter}(t)$")

    ax1.axis("off")
    ax1.set_aspect("equal")

    ax1.set_xlim((-5, 6))
    ax1.set_ylim((-5, 8))

    real_axis = Arrow(Point(-5, 0), Point(6, 0)).drawn(ax1)
    real_axis.end.labeled(ax1, r"\rm Re", (7.5, -3))
    imag_axis = Arrow(Point(0, -5), Point(0, 6.5)).drawn(ax1)
    imag_axis.end.labeled(ax1, r"\rm Im", (0, 2.5))

    ts = np.linspace(0, 2 * np.pi)

    origin = Point(0, 0)
    origin.drawn(ax1)
    for i, angle in enumerate(angles):
        arrow = Arrow.from_polar(origin, length=4, angle=angle).drawn(
            ax1, linewidth=1.5
        )
        phase_letter = "abcde"[i]
        arrow.rotated(0.3).end.labeled(ax1, f"$V_{phase_letter}$")
        label = f"${phase_letter}$"
        ax2.plot(ts, np.cos(ts + angle))
        ax3.plot(ts, np.square(np.cos(ts + angle)), label=label)

    ax2.set_ylim((-1.1, 1.1))

    ax3.plot(
        ts,
        sum(np.square(np.cos(ts + angle)) for angle in angles),
        "k",
        label=r"\rm total",
    )
    ax3.set_ylim((-0.1, 3))

    for ax in [ax2, ax3]:
        ax.set_xlim((0, 2 * np.pi))
        ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
        ax.set_xticklabels(["$0$", r"$\pi / 2$", r"$\pi$", r"$3 \pi / 2$", r"$2 \pi$"])

    ax3.legend(loc="upper right", prop=dict(size=6))

    if angles == [0.0]:
        ax2.set_title(r"\small $v_i(t) / |V|$")
        ax3.set_title(r"\small $p_i(t) / (|V|^2 / R)$")


def generate_phase_angles(n: int) -> list[float]:
    return cast(list[float], np.linspace(0, 2 * np.pi, n + 1).tolist())[:-1]


fig, axs = plt.subplots(nrows=5, ncols=4, figsize=(6.5, 9), layout="constrained")
# Single-phase:
draw(*axs[0], angles=[0.0])
# Two-phase:
draw(*axs[1], angles=[0.0, np.pi / 2])
# Three-phase:
draw(*axs[2], angles=generate_phase_angles(3))
# Four-phase:
draw(*axs[3], angles=generate_phase_angles(4))
# Five-phase:
draw(*axs[4], angles=generate_phase_angles(5))
fig.suptitle(r"\small \rm Figure 5.1")

fig.savefig("img/fig_5_1.png", dpi=200)

```{image} img/fig_5_1.png
:align: center
:width: 100%
```

<p></p>